# 🧠 Tutorial 3 — EEG Analysis & Classification
### 4th Year · Biomedical Signals & Applications · EEG Python Workshop

---

**Goal of this tutorial:** Analyse the clean EEG epochs and build a classifier that can distinguish left-hand from right-hand motor imagery.

**By the end you will be able to:**
- Compute and interpret Event-Related Potentials (ERPs)
- Analyse the power spectrum and extract band power features
- Build time-frequency maps and identify ERD/ERS patterns
- Construct a feature matrix for machine learning
- Train and evaluate an LDA classifier with cross-validation
- Apply CSP spatial filtering for improved classification

**Time:** approximately 60 minutes

---

> ⚠️ **Run Tutorial 2 first** — it creates the clean epoch file this notebook loads.

---
## Setup — Imports

Run this cell first every time you open the notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import mne
from mne.time_frequency import tfr_morlet
from scipy.signal import welch
from scipy.stats import ttest_ind

mne.set_log_level("WARNING")
print("All imports successful ✓")

---
## Step 0 — Load Preprocessed Epochs

We load the clean epochs saved in Tutorial 2 and split them by condition (left hand vs right hand).

In [ ]:
epochs = mne.read_epochs("/tmp/motor_imagery_epochs-epo.fif", verbose=False)

# Split by condition
epochs_left  = epochs["left_hand"]
epochs_right = epochs["right_hand"]

print("Epochs loaded ✓")
print(f"  Left hand  : {len(epochs_left)} trials")
print(f"  Right hand : {len(epochs_right)} trials")
print(f"  Channels   : {len(epochs.ch_names)}")
print(f"  Time axis  : {epochs.times[0]:.1f} s  to  {epochs.times[-1]:.1f} s")
print(f"  Shape      : {epochs.get_data().shape}  (trials × channels × time points)")

---
## Step 1 — Event-Related Potentials (ERPs)

### What is an ERP?

An **ERP** is the average of many EEG epochs time-locked to the same event. Random background noise averages out (positive and negative cancel each other). Only the brain's consistent, time-locked response survives.

> 🟢 **Analogy:** Photographing a faint star in the night sky. One 1-second exposure shows mostly noise. Average 100 photos and the noise cancels — the star becomes clearly visible.

### The contralateral rule

The motor cortex is organised **contralaterally** — the LEFT hemisphere controls the RIGHT hand:

| Electrode | Brain region | Most active during |
|---|---|---|
| **C3** | Left motor cortex | **Right** hand imagery |
| **Cz** | Central / bilateral | Legs, bilateral |
| **C4** | Right motor cortex | **Left** hand imagery |

The C3 vs C4 comparison should show this lateralisation directly.

In [ ]:
# Average all trials within each condition
evoked_left  = epochs_left.average()
evoked_right = epochs_right.average()

print(f"Evoked left  : {evoked_left.nave} trials averaged")
print(f"Evoked right : {evoked_right.nave} trials averaged")

In [ ]:
# Butterfly plot — all channels, both conditions
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
evoked_left.plot( axes=axes[0], show=False, titles="Evoked — Left Hand Imagery")
evoked_right.plot(axes=axes[1], show=False, titles="Evoked — Right Hand Imagery")
plt.tight_layout()
plt.show()

In [ ]:
# Topographic maps at several time points during imagery
times_topo = np.arange(0.5, 3.0, 0.5)   # 0.5, 1.0, 1.5, 2.0, 2.5 s
fig = evoked_left.plot_topomap(times=times_topo, ch_type="eeg", show=True)
plt.suptitle("Left Hand Imagery — Scalp Distribution Over Time")
plt.show()

In [ ]:
# Compare C3 and C4 for both conditions — this is the lateralisation plot
idx_c3 = epochs.ch_names.index("C3")
idx_c4 = epochs.ch_names.index("C4")

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, ch_idx, ch_name in zip(axes, [idx_c3, idx_c4], ["C3", "C4"]):
    ax.plot(evoked_left.times,  evoked_left.data[ch_idx]  * 1e6,
            "steelblue", lw=2, label="Left hand imagery")
    ax.plot(evoked_right.times, evoked_right.data[ch_idx] * 1e6,
            "tomato",    lw=2, label="Right hand imagery")
    ax.axvline(0, color="k", lw=1, linestyle="--", label="Cue onset")
    ax.axhline(0, color="k", lw=0.5)
    ax.set_title(f"ERP at {ch_name}")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Amplitude (µV)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
plt.suptitle("Lateralised Motor Response: C3 (left hemisphere) vs C4 (right hemisphere)")
plt.tight_layout()
plt.show()

> 👁 **What to check:**
> - C3 and C4 should show **opposite patterns** — when C3 is more negative for one condition, C4 should be more negative for the other
> - This is the contralateral lateralisation: right-hand imagery activates C3 (left hemisphere), left-hand activates C4 (right hemisphere)
> - The difference between conditions builds up gradually after the cue (t = 0)

---
## Step 2 — Power Spectral Density (PSD) & Band Power

### EEG frequency bands

Different mental states produce different dominant frequencies:

| Band | Range | What it means for motor imagery |
|---|---|---|
| **Delta** | 0.5–4 Hz | Deep sleep (not relevant here) |
| **Theta** | 4–8 Hz | Memory, frontal activity |
| **Alpha / Mu** | 8–13 Hz | **Motor cortex idle rhythm — decreases during imagery (ERD)** |
| **Beta** | 13–30 Hz | **Active motor processing — also decreases during imagery** |
| **Gamma** | 30–45 Hz | Higher cognition; muscle noise overlaps here |

> 🟢 **Analogy:** The PSD is like a music equaliser — it shows the "volume" at each frequency. We are looking for whether the "mu/alpha volume knob" is turned down during motor imagery.

### Welch's method

`compute_psd(method='welch')` splits the signal into overlapping windows, computes the FFT of each, and averages them. This gives a much smoother spectrum than a single FFT.

In [ ]:
# Plot Welch PSD for a single epoch, channel C3
fs = epochs.info["sfreq"]
ep_data = epochs_left.get_data()   # (n_epochs, n_channels, n_times)

freqs_w, psd_w = welch(ep_data[0, idx_c3], fs=fs, nperseg=int(fs * 2))

fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(freqs_w, psd_w, "steelblue", lw=1.5)
ax.set_xlim(0, 50)
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("PSD (V²/Hz)")
ax.set_title("Welch PSD — Channel C3, Left Hand (single epoch)")

# Shade the EEG bands
band_colors = {"Delta": "#c0e8ff", "Theta": "#a0d8a0",
               "Alpha/Mu": "#ffe0a0", "Beta": "#ffc0c0", "Gamma": "#e0c0ff"}
band_limits = {"Delta": (0.5,4), "Theta": (4,8), "Alpha/Mu": (8,13),
               "Beta": (13,30), "Gamma": (30,45)}
for band, (lo, hi) in band_limits.items():
    ax.axvspan(lo, hi, alpha=0.3, color=band_colors[band], label=band)
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Compute mean PSD over all trials using MNE — better estimate
psd_left  = epochs_left.compute_psd( method="welch", fmin=1, fmax=45)
psd_right = epochs_right.compute_psd(method="welch", fmin=1, fmax=45)

def band_power(psd_obj, fmin, fmax):
    """Average PSD power within a frequency band. Returns (n_epochs, n_channels)."""
    data, freqs = psd_obj.get_data(return_freqs=True)
    mask = (freqs >= fmin) & (freqs <= fmax)
    return data[:, :, mask].mean(axis=-1)

# Compare alpha/mu band power at C3 between conditions
bp_left_alpha  = band_power(psd_left,  8, 13)   # (n_epochs, 64)
bp_right_alpha = band_power(psd_right, 8, 13)

print("Alpha/Mu band power at C3:")
print(f"  Left hand  : {bp_left_alpha[:,  idx_c3].mean():.4e} V²/Hz")
print(f"  Right hand : {bp_right_alpha[:, idx_c3].mean():.4e} V²/Hz")

diff_pct = (bp_left_alpha[:, idx_c3].mean() - bp_right_alpha[:, idx_c3].mean()) /             bp_left_alpha[:, idx_c3].mean() * 100
print(f"  Difference : {diff_pct:.1f}%  (positive = more power for left hand)")

> 👁 **What to check:**
> - The alpha/mu power at C3 should be **lower for right-hand** imagery (ERD is stronger at C3 during contralateral imagery)
> - If the difference is near zero, this subject may not show strong lateralisation — try C4

---
## Step 3 — Time-Frequency Analysis (TFR) & ERD/ERS

### Why time-frequency?

A power spectrum averages over the whole epoch — it cannot tell you *when* the frequencies changed. A **time-frequency representation (TFR)** is a 2D image showing how power at each frequency evolves over time.

> 🟢 **Analogy:** A PSD is a recipe card listing all ingredients. A TFR is a video of the cooking process — you can see when each ingredient was added and how the mixture changed over time.

### ERD and ERS

| Term | Meaning | What it looks like in the TFR |
|---|---|---|
| **ERD** (Desynchronisation) | Power **decreases** below baseline | **Blue** region in the TFR |
| **ERS** (Synchronisation) | Power **increases** above baseline | **Red** region in the TFR |

**Expected pattern for right-hand imagery at C3:**
- Blue patch at 8–13 Hz starting ~0.5 s after cue (mu ERD)
- Blue patch at 18–26 Hz over the same period (beta ERD)
- Possible red patch at 18–26 Hz after 3 s (beta rebound — motor cortex returning to idle)

### Morlet wavelets

`tfr_morlet()` slides short wavelets at each frequency along the signal and measures how well they match. The parameter `n_cycles` controls the trade-off between time and frequency precision.

In [ ]:
freqs_tfr = np.arange(6, 40, 1)    # analyse 6 to 40 Hz in 1 Hz steps
n_cycles  = freqs_tfr / 2.0         # wavelet length: frequency/2 cycles (balanced trade-off)

print(f"Computing TFR for {len(freqs_tfr)} frequencies × {len(epochs.times)} time points...")

tfr_left  = tfr_morlet(epochs_left,  freqs=freqs_tfr, n_cycles=n_cycles,
                        use_fft=True, return_itc=False, verbose=False)
tfr_right = tfr_morlet(epochs_right, freqs=freqs_tfr, n_cycles=n_cycles,
                        use_fft=True, return_itc=False, verbose=False)

# Express power as percentage change from the pre-stimulus baseline (−0.5 to 0 s)
tfr_left.apply_baseline( (-0.5, 0), mode="percent", verbose=False)
tfr_right.apply_baseline((-0.5, 0), mode="percent", verbose=False)

print("TFR computed ✓")

In [ ]:
# Plot TFR at C3 for both conditions side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
tfr_left.plot( [idx_c3], axes=axes[0], show=False,
               title="TFR at C3 — Left Hand Imagery (% change from baseline)")
tfr_right.plot([idx_c3], axes=axes[1], show=False,
               title="TFR at C3 — Right Hand Imagery (% change from baseline)")
plt.tight_layout()
plt.show()

> 👁 **What to check:**
> - For **right-hand imagery** at C3: expect a stronger blue region (bigger ERD) in the mu (8–13 Hz) and beta (18–26 Hz) bands
> - For **left-hand imagery** at C3: less ERD (C3 is *ipsilateral* to the left hand — the opposite hemisphere activates)
> - This asymmetry is the core signal that makes motor imagery BCI work

In [ ]:
# Difference map: right hand minus left hand at C3
# Red = more power for right; Blue = more power for left
tfr_diff = tfr_right.data[idx_c3] - tfr_left.data[idx_c3]

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(
    tfr_diff,
    aspect="auto",
    origin="lower",
    extent=[epochs.times[0], epochs.times[-1], freqs_tfr[0], freqs_tfr[-1]],
    cmap="RdBu_r",
    vmin=-0.5, vmax=0.5,
)
ax.axvline(0, color="k", lw=1.5, linestyle="--", label="Cue onset")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Frequency (Hz)")
ax.set_title("TFR Difference (Right − Left) at C3\nRed = more power for right; Blue = more power for left")
ax.legend()
plt.colorbar(im, ax=ax, label="Relative Power Difference")
plt.tight_layout()
plt.show()

---
## 🧠 Interlude: The Motor Cortex Story

Before we extract features, let's make sure we understand what the TFR is showing us.

### The mu rhythm — the motor cortex's idle state

The motor cortex produces a rhythmic oscillation at 8–12 Hz when it is at rest — called the **mu rhythm** (related to the alpha rhythm). When you move your hand — or even *imagine* moving it — the mu rhythm disappears. This disappearance is the **ERD** (Event-Related Desynchronisation).

> 🟢 **Analogy:** The mu rhythm is like a car engine idling in neutral. When the driver decides to move — or even just *thinks* about moving — the engine engages and the rhythmic idle stops. When the driver parks again, the idle resumes (sometimes briefly overshooting — that is the beta rebound).

### Why the lateralisation matters for BCI

```
Right-hand imagery  →  LEFT motor cortex activates  →  C3 shows ERD
Left-hand imagery   →  RIGHT motor cortex activates →  C4 shows ERD
```

By comparing the mu/beta power at C3 and C4, a computer can predict which hand the subject was imagining — **without any movement happening**. This is the core of motor imagery BCI.

---
## Step 4 — Feature Extraction for Machine Learning

### From a matrix to a vector

Each epoch is a 2D matrix: **64 channels × 561 time points = 35,904 numbers**. A classifier cannot work with this raw matrix. We need to summarise each epoch as a small set of informative numbers — a **feature vector**.

Here we extract:
- Alpha/mu band power (8–13 Hz) at 5 channels: C3, Cz, C4, FC3, FC4
- Beta band power (13–30 Hz) at the same 5 channels

This gives **10 features per trial** — much more manageable than 35,904.

> 🟢 **Analogy:** Summarising a 2-hour football match with statistics: goals, possession %, shots on target. You cannot watch 2 hours to compare 80 matches — but 5 numbers per match tell you what matters.

In [ ]:
def extract_band_power_features(epochs_obj, channels, bands_dict):
    """
    Build a feature matrix from band-power across selected channels.

    Returns
    -------
    X            : np.ndarray  shape (n_epochs, n_features)
    feature_names: list of str
    """
    psd = epochs_obj.compute_psd(method="welch", fmin=1, fmax=45)
    data_psd, freqs = psd.get_data(return_freqs=True)

    ch_idx = [epochs_obj.ch_names.index(c) for c in channels]
    features, feature_names = [], []

    for band, (fmin, fmax) in bands_dict.items():
        mask = (freqs >= fmin) & (freqs <= fmax)
        bp   = data_psd[:, :, mask].mean(axis=-1)   # (n_epochs, n_channels)
        for ci, ch_name in zip(ch_idx, channels):
            features.append(bp[:, ci])
            feature_names.append(f"{band}_{ch_name}")

    return np.column_stack(features), feature_names


sel_channels = ["C3", "Cz", "C4", "FC3", "FC4"]
sel_bands    = {"Alpha": (8, 13), "Beta": (13, 30)}

X_left,  feat_names = extract_band_power_features(epochs_left,  sel_channels, sel_bands)
X_right, _          = extract_band_power_features(epochs_right, sel_channels, sel_bands)

X = np.vstack([X_left,  X_right])
y = np.hstack([np.zeros(len(X_left)), np.ones(len(X_right))])  # 0=left, 1=right

print(f"Feature matrix X shape : {X.shape}  (trials × features)")
print(f"Labels y shape         : {y.shape}")
print(f"Feature names: {feat_names}")

In [ ]:
# Distribution plot — how separable are the features?
fig, axes = plt.subplots(2, len(sel_channels), figsize=(14, 6))

for row_i, band in enumerate(sel_bands):
    for col_i, ch in enumerate(sel_channels):
        feat_label = f"{band}_{ch}"
        fi  = feat_names.index(feat_label)
        ax  = axes[row_i, col_i]

        ax.hist(X[y == 0, fi], bins=12, alpha=0.6, label="Left",  color="steelblue")
        ax.hist(X[y == 1, fi], bins=12, alpha=0.6, label="Right", color="tomato")

        # t-test: is this feature significantly different between conditions?
        t_stat, p_val = ttest_ind(X[y == 0, fi], X[y == 1, fi])
        title_color = "darkred" if p_val < 0.05 else "black"
        marker = " *" if p_val < 0.05 else ""
        ax.set_title(f"{feat_label}{marker}", fontsize=8, color=title_color)
        if col_i == 0:
            ax.set_ylabel(f"{band} Power")
        ax.legend(fontsize=6)

plt.suptitle("Feature Distributions: Left (blue) vs Right (red) Hand Imagery\n* = statistically significant (p < 0.05)", fontsize=10)
plt.tight_layout()
plt.show()

> 👁 **What to check:**
> - Features marked with **\*** are statistically different between conditions
> - Features at **C3** and **C4** should show the clearest separation (lateralised ERD)
> - Overlapping histograms mean the feature alone is not enough — we need to combine features

---
## Step 5 — LDA Classification

### What is LDA?

**Linear Discriminant Analysis (LDA)** finds the straight line (in 2D) or hyperplane (in 10D) that best separates the two classes. It maximises the distance between the class means while minimising the spread within each class.

> 🟢 **Analogy:** Two piles of gold and silver coins mixed on a table. LDA finds the dividing line that puts as many gold coins on one side and silver coins on the other as possible.

### Why log-transform first?

Band power values have a skewed distribution — a few trials with very high power. LDA assumes Gaussian distributions. The log transform compresses large values and makes the distribution more symmetric.

### Cross-validation — honest accuracy

Never test on the same data you trained on! **5-fold cross-validation** splits data into 5 groups; trains on 4, tests on 1, rotates 5 times. Every trial is used as a test exactly once. The final accuracy is the average.

| Accuracy | Interpretation |
|---|---|
| ~50% | Chance level (guessing randomly) |
| 55–65% | Weak signal — features need improvement |
| 65–75% | Typical for band power + LDA |
| 75–85% | Good — clear ERD present |
| 85%+ | Excellent |

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Log-transform: makes distribution more Gaussian → better for LDA
X_log = np.log10(X + 1e-30)

pipeline = Pipeline([
    ("scaler", StandardScaler()),              # zero-mean, unit-variance per feature
    ("lda",    LinearDiscriminantAnalysis()),  # find the best separating line
])

cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X_log, y, cv=cv, scoring="accuracy")

print("LDA cross-validation results:")
print(f"  Per-fold accuracies : {[f'{s:.1%}' for s in scores]}")
print(f"  Mean accuracy       : {scores.mean():.1%}  ±  {scores.std():.1%}")
print(f"  Chance level        : 50.0%")

if scores.mean() > 0.60:
    print("  ✓  Above chance — the classifier is learning the lateralisation pattern")
else:
    print("  ⚠️  Near chance — check that TFR shows clear ERD difference")

In [ ]:
# Decision boundary visualisation (using the first two features for 2D plot)
pipeline.fit(X_log, y)

f0, f1 = 0, 1  # Alpha_C3 and Alpha_Cz
x_min, x_max = X_log[:, f0].min() - 0.5, X_log[:, f0].max() + 0.5
y_min, y_max = X_log[:, f1].min() - 0.5, X_log[:, f1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                     np.linspace(y_min, y_max, 200))

# Fill the remaining dimensions with zeros for visualisation
grid_pts = np.c_[xx.ravel(), yy.ravel(),
                 np.zeros((xx.size, X_log.shape[1] - 2))]
Z = pipeline.predict(grid_pts).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(8, 6))
ax.contourf(xx, yy, Z, alpha=0.25, cmap="bwr")
ax.scatter(X_log[y==0, f0], X_log[y==0, f1],
           c="steelblue", edgecolors="k", lw=0.5, label="Left hand",  s=50)
ax.scatter(X_log[y==1, f0], X_log[y==1, f1],
           c="tomato",    edgecolors="k", lw=0.5, label="Right hand", s=50)
ax.set_xlabel(f"log₁₀ [{feat_names[f0]}]")
ax.set_ylabel(f"log₁₀ [{feat_names[f1]}]")
ax.set_title(f"LDA Decision Boundary (2D projection)\nCV accuracy = {scores.mean():.1%}")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Step 6 — CSP + LDA: The Motor Imagery Gold Standard

### What is CSP?

In Step 4 we manually chose which channels to use (C3, Cz, C4, FC3, FC4). **CSP (Common Spatial Patterns)** finds the *optimal* combination of all 64 channels automatically from the data.

CSP creates **virtual channels** (spatial filters) that:
- **Maximise** variance for one condition (e.g. left hand)
- **Minimise** variance for the other condition (right hand)

The features extracted from CSP components are much better at separating the two classes than manually chosen band power.

> 🟢 **Analogy:** Instead of you choosing which microphone to use in a noisy room, CSP finds the ideal *mix* of all microphones that best picks up the signal you want.

### Why CSP + LDA is the standard

CSP + LDA has been the dominant motor imagery BCI method since the mid-2000s. Despite being simple, it consistently outperforms hand-crafted features in BCI competitions.

In [ ]:
from mne.decoding import CSP
from sklearn.pipeline import Pipeline as SKPipeline

csp = CSP(n_components=4, log=True, reg=None)

pipeline_csp = SKPipeline([
    ("csp", csp),
    ("lda", LinearDiscriminantAnalysis()),
])

# CSP takes raw epoch data (not pre-computed features)
X_raw = np.vstack([epochs_left.get_data(), epochs_right.get_data()])

scores_csp = cross_val_score(pipeline_csp, X_raw, y, cv=cv, scoring="accuracy")

print("CSP + LDA cross-validation results:")
print(f"  Per-fold accuracies : {[f'{s:.1%}' for s in scores_csp]}")
print(f"  Mean accuracy       : {scores_csp.mean():.1%}  ±  {scores_csp.std():.1%}")
print(f"\nComparison:")
print(f"  Band power LDA : {scores.mean():.1%}")
print(f"  CSP + LDA      : {scores_csp.mean():.1%}")
print(f"  Improvement    : {(scores_csp.mean() - scores.mean())*100:+.1f} percentage points")

In [ ]:
# Fit CSP on full dataset and plot spatial patterns
pipeline_csp.fit(X_raw, y)

fig = csp.plot_patterns(epochs.info, show=True,
                        title="CSP Spatial Patterns — where does each component come from?")
plt.show()

> 👁 **What to check in the CSP patterns:**
> - **Patterns 1–2** (maximise left-hand variance): focal activation near **C4** (right hemisphere)
> - **Patterns 3–4** (maximise right-hand variance): focal activation near **C3** (left hemisphere)
> - If patterns show no clear spatial structure, the dataset may not have enough trials or the ERD is weak
>
> Note: we are plotting `plot_patterns()` not `plot_filters()`. The patterns show *where the signal comes from* on the scalp — that is what has neurological meaning.

---
## ✅ Tutorial 3 — Summary

| Step | MNE / Python Command | What it produces |
|---|---|---|
| 0 — Load | `mne.read_epochs()` | Clean epochs split by condition |
| 1 — ERP | `epochs.average()`, `plot_topomap()` | Average waveform + scalp maps |
| 2 — PSD | `compute_psd()`, `band_power()` | Power spectrum + band power values |
| 3 — TFR | `tfr_morlet()`, `apply_baseline()` | Time × frequency ERD/ERS map |
| 4 — Features | `extract_band_power_features()` | 10-feature matrix per trial |
| 5 — LDA | `Pipeline([StandardScaler, LDA])` | Accuracy via 5-fold CV |
| 6 — CSP + LDA | `CSP()` + `Pipeline` | Better accuracy + spatial patterns |

---

### Key findings in motor imagery EEG

- **ERD** (power decrease) in the mu (8–12 Hz) and beta (18–26 Hz) bands during imagery
- ERD is **lateralised**: right-hand imagery → stronger ERD at C3 (left hemisphere)
- **CSP + LDA** consistently outperforms hand-picked band power features
- Typical accuracy: 65–80% for this single-subject dataset

---

### 🏆 The complete BCI pipeline

```
Raw EEG (Tutorial 1)
    ↓
Preprocessing: filter, ICA, epoch (Tutorial 2)
    ↓
ERD analysis: TFR at C3/C4 (Tutorial 3, Steps 1–3)
    ↓
Feature extraction: band power or CSP log-variance (Step 4–6)
    ↓
Classification: LDA with cross-validation (Step 5–6)
    ↓
Prediction: left or right hand imagery from new EEG
```

This pipeline is the foundation of commercial EEG-based BCI systems used clinically for motor rehabilitation and assistive communication.